# Digit addition: a from-scratch transformer, and its addition circuit

## What this project is

Train a small transformer from scratch on **decimal addition**, then reverse-engineer the
algorithm it learns. This extends Nanda et al.'s *Progress measures for grokking via
mechanistic interpretability*, which analysed **modular** arithmetic and found the model had
learned a Fourier/trigonometric algorithm. Ordinary multi-digit addition is a different problem
-- it is positional and needs carrying -- so whatever circuit appears should be structurally
different, and that difference is the contribution.

## Why it is set up this way

Notebook 1 (`1_transformer_from_scratch.ipynb`) was the precursor: build a GPT-2-shaped
transformer by hand and prove it trains. That is done. Everything below the "carried over"
heading is that code, unchanged apart from dropping ARENA's test harness.

The important change here is the **vocabulary**: 12 symbols (`0`-`9`, `+`, `=`) instead of
50257. In notebook 1, 97% of the parameters were the embedding table and only 3% did any
computation. Here it inverts -- essentially every parameter computes. That is what makes the
interpretability tractable: there is an actual algorithm inside, not a lookup table.

## The minimum result

**Identify which attention heads implement the carry, via ablation plus attention patterns.**

Anything beyond that -- the full mechanism, a comparison of circuits across digit positions,
grokking dynamics -- is bonus. This phase has no natural end, so the floor matters.


In [ ]:
import math
from dataclasses import dataclass

import einops
import numpy as np
import torch as t
import torch.nn as nn
import wandb
from jaxtyping import Float, Int
from torch import Tensor
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm
from transformer_lens import HookedTransformer, HookedTransformerConfig
from transformer_lens.utils import gelu_new

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
print("device:", device)

## Carried over from notebook 1

Your implementations, unchanged. ARENA's `tests.*` calls have been stripped since the
exercises directory isn't on the path here.

In [ ]:
@dataclass
class Config:
    d_model: int = 768
    debug: bool = True
    layer_norm_eps: float = 1e-5
    d_vocab: int = 50257
    init_range: float = 0.02
    n_ctx: int = 1024
    d_head: int = 64
    d_mlp: int = 3072
    n_heads: int = 12
    n_layers: int = 12

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.w = nn.Parameter(t.ones(cfg.d_model))
        self.b = nn.Parameter(t.zeros(cfg.d_model))

    def forward(self, residual: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
        # residual: [batch, position, d_model]
        if self.cfg.debug: print("Residual: ", residual.shape)
        residual_mean = residual.mean(dim=-1, keepdim=True)
        residual_std = (residual.var(dim=-1, keepdim=True, correction=0) + self.cfg.layer_norm_eps).sqrt()

        normalised = (residual - residual_mean) / residual_std
        if self.cfg.debug: print("Normalised: ", normalised.shape)
        return normalised * self.w + self.b

In [ ]:
class Embed(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_E = nn.Parameter(t.empty((cfg.d_vocab, cfg.d_model)))
        nn.init.normal_(self.W_E, std=self.cfg.init_range)

    def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_model"]:
        return self.W_E[tokens]

In [ ]:
class PosEmbed(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_pos = nn.Parameter(t.empty((cfg.n_ctx, cfg.d_model)))
        nn.init.normal_(self.W_pos, std=self.cfg.init_range)

    def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_model"]:
        batch, seq_len = tokens.shape
        return einops.repeat(self.W_pos[:seq_len], "position embedding -> batch position embedding", batch=batch)

In [ ]:
class Attention(nn.Module):
    IGNORE: Float[Tensor, ""]

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_Q = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.W_K = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.W_V = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.W_O = nn.Parameter(t.empty((cfg.n_heads, cfg.d_head, cfg.d_model)))
        self.b_Q = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))
        self.b_K = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))
        self.b_V = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))
        self.b_O = nn.Parameter(t.zeros((cfg.d_model)))
        nn.init.normal_(self.W_Q, std=self.cfg.init_range)
        nn.init.normal_(self.W_K, std=self.cfg.init_range)
        nn.init.normal_(self.W_V, std=self.cfg.init_range)
        nn.init.normal_(self.W_O, std=self.cfg.init_range)
        self.register_buffer("IGNORE", t.tensor(float("-inf"), dtype=t.float32, device=device))

    def forward(self, normalized_resid_pre: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
        Q = einops.einsum(
            normalized_resid_pre,
            self.W_Q,
            "batch posn d_model, n_heads d_model d_head -> batch posn n_heads d_head"
        ) + self.b_Q
        
        K = einops.einsum(
            normalized_resid_pre,
            self.W_K,
            "batch posn d_model, n_heads d_model d_head -> batch posn n_heads d_head"
        ) + self.b_K
        
        V = einops.einsum(
                    normalized_resid_pre,
                    self.W_V,
                    "batch posn d_model, n_heads d_model d_head -> batch posn n_heads d_head"
                ) + self.b_V
        
        attn_scores = einops.einsum(
            Q,
            K,
            "batch posn_Q n_heads d_head, batch posn_K n_heads d_head -> batch n_heads posn_Q posn_K"
        )
        
        attn_scores_masked = self.apply_causal_mask(attn_scores / self.cfg.d_head**0.5)
        
        attn_pattern = attn_scores_masked.softmax(-1)
        
        z = einops.einsum(
            V,
            attn_pattern,
            "batch posn_K n_heads d_head, batch n_heads posn_Q posn_K -> batch posn_Q n_heads d_head"
        )
        
        result = einops.einsum(
            self.W_O,
            z,
            "n_heads d_head d_model, batch posn n_heads d_head -> batch posn n_heads d_model"
        )
        
        attn_out = einops.reduce(
            result, 
            "batch posn n_heads d_model -> batch posn d_model",
            'sum'
        ) + self.b_O
        
        return attn_out
        
    def apply_causal_mask(
        self, attn_scores: Float[Tensor, "batch n_heads query_pos key_pos"]
    ) -> Float[Tensor, "batch n_heads query_pos key_pos"]:
        """
        Applies a causal mask to attention scores, and returns masked scores.
        """
        query_pos, key_pos = attn_scores.shape[-2:]
        all_ones = t.ones(query_pos, key_pos, dtype=t.bool, device=attn_scores.device)
        future_positions = t.triu(all_ones, diagonal=1)
        return attn_scores.masked_fill(future_positions, self.IGNORE)

In [ ]:
class MLP(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_in = nn.Parameter(t.empty((cfg.d_model, cfg.d_mlp)))
        self.W_out = nn.Parameter(t.empty((cfg.d_mlp, cfg.d_model)))
        self.b_in = nn.Parameter(t.zeros((cfg.d_mlp)))
        self.b_out = nn.Parameter(t.zeros((cfg.d_model)))
        self.act = gelu_new         # new addition
        nn.init.normal_(self.W_in, std=self.cfg.init_range)
        nn.init.normal_(self.W_out, std=self.cfg.init_range)

    def forward(self, normalized_resid_mid: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
        pre = einops.einsum(
            normalized_resid_mid,
            self.W_in,
            "batch posn d_model, d_model d_mlp -> batch posn d_mlp"
        ) + self.b_in
        
        post = self.act(pre)
        
        mlp_out = einops.einsum(
            post,
            self.W_out,
            "batch posn d_mlp, d_mlp d_model -> batch posn d_model"
        ) + self.b_out
        
        return mlp_out

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.ln1 = LayerNorm(cfg)
        self.attn = Attention(cfg)
        self.ln2 = LayerNorm(cfg)
        self.mlp = MLP(cfg)

    def forward(self, resid_pre: Float[Tensor, "batch position d_model"]) -> Float[Tensor, "batch position d_model"]:
        resid_mid = self.attn(self.ln1(resid_pre)) + resid_pre
        resid_post = self.mlp(self.ln2(resid_mid)) + resid_mid
        return resid_post

In [ ]:
class Unembed(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.W_U = nn.Parameter(t.empty((cfg.d_model, cfg.d_vocab)))
        nn.init.normal_(self.W_U, std=self.cfg.init_range)
        self.b_U = nn.Parameter(t.zeros((cfg.d_vocab)), requires_grad=False)

    def forward(
        self, normalized_resid_final: Float[Tensor, "batch position d_model"]
    ) -> Float[Tensor, "batch position d_vocab"]:
        logits = normalized_resid_final @ self.W_U + self.b_U
        return logits

In [ ]:
class DemoTransformer(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.embed = Embed(cfg)
        self.pos_embed = PosEmbed(cfg)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_final = LayerNorm(cfg)
        self.unembed = Unembed(cfg)

    def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_vocab"]:
        residual_stream = self.embed(tokens) + self.pos_embed(tokens)
        for block in self.blocks:
            residual_stream = block(residual_stream)
        logits = self.unembed(self.ln_final(residual_stream))
        return logits

In [ ]:
def get_log_probs(
    logits: Float[Tensor, "batch posn d_vocab"], tokens: Int[Tensor, "batch posn"]
) -> Float[Tensor, "batch posn-1"]:
    log_probs = logits.log_softmax(dim=-1)
    # Get logprobs the first seq_len-1 predictions (so we can compare them with the actual next tokens)
    log_probs_for_tokens = log_probs[:, :-1].gather(dim=-1, index=tokens[:, 1:].unsqueeze(-1)).squeeze(-1)

    return log_probs_for_tokens

### Training harness

Two changes from notebook 1, both because this project has no tokenizer and no text sampling:

- `TransformerTrainer` now takes `dataset_dict` as an argument rather than reading a global.
- The `TransformerSampler` and its per-epoch text samples are gone. You'll want a
  task-specific eval instead -- exact-match accuracy on whole answers, not next-token accuracy.

`training_step` and `evaluate` are untouched.

In [ ]:
@dataclass
class TransformerTrainingArgs:
    batch_size: int = 32
    epochs: int = 10
    max_steps_per_epoch: int = 500
    lr: float = 1e-3
    weight_decay: float = 1e-2
    wandb_project: str | None = "day1-demotransformer"
    wandb_name: str | None = None
    eval_prompt: str = "Once upon a time"

In [ ]:
class TransformerTrainer:
    def __init__(self, args: TransformerTrainingArgs, model: DemoTransformer, dataset_dict):
        super().__init__()
        self.model = model
        self.args = args
        self.optimizer = t.optim.AdamW(self.model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
        self.step = 0

        self.train_loader = DataLoader(
            dataset_dict["train"],
            batch_size=args.batch_size,
            shuffle=True,
            num_workers=4, #runs faster, but kills the kernel if interrupted
            #pin_memory=False,
        )
        self.test_loader = DataLoader(
            dataset_dict["test"],
            batch_size=args.batch_size,
            shuffle=False,
            num_workers=4,
            #pin_memory=False,
        )

    def training_step(self, batch: dict[str, Int[Tensor, "batch seq"]]) -> Float[Tensor, ""]:
        """
        Calculates the loss on the tokens in the batch, performs a gradient update step, and logs the loss.

        Remember that `batch` is a dictionary with the single key 'tokens'.
        """
        tokens = batch["tokens"].to(device)
        logits = self.model(tokens)
        loss = -get_log_probs(logits, tokens).mean()
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()
        self.step += 1
        wandb.log({"train_loss": loss}, step=self.step)
        return loss

    @t.inference_mode()
    def evaluate(self) -> float:
        """
        Evaluate the model on the test set and return the accuracy.
        """
        self.model.eval()
        total_correct, total_predicted = 0, 0
        for batch in self.test_loader:
            tokens = batch["tokens"].to(device)
            logits: Tensor = self.model(tokens)[:, :-1]
            predicted_tokens = logits.argmax(dim=-1)
            total_correct += (predicted_tokens == tokens[:, 1:]).sum().item()
            total_predicted += tokens.shape[0] * (tokens.shape[-1] - 1)
        
        accuracy = total_correct / total_predicted
        self.model.train()
        wandb.log({"accuracy": accuracy}, step=self.step)
        return accuracy

    def train(self):
        """
        Trains the model, for `self.args.epochs` epochs. Also handles wandb initialisation, and early stopping
        for each epoch at `self.args.max_steps_per_epoch` steps.
        """
        wandb.init(project=self.args.wandb_project, name=self.args.wandb_name, config=self.args)
        accuracy = np.nan

        progress_bar = tqdm(total=self.args.max_steps_per_epoch * self.args.epochs)
        for epoch in range(self.args.epochs):
            for i, batch in enumerate(self.train_loader):
                loss = self.training_step(batch)
                progress_bar.update()
                progress_bar.set_description(f"Epoch {epoch + 1}, loss: {loss:.3f}, accuracy: {accuracy:.3f}")
                if i > self.args.max_steps_per_epoch:
                    break

            accuracy = self.evaluate()

        wandb.finish()

---

# What to build next

## 1. The task encoding

Decide and write down:

- **Vocabulary** -- `0`-`9`, `+`, `=` is 12 symbols. A dict, not a trained tokenizer.
- **Sequence format** -- e.g. `49+97=641`, fixed length so every example is the same shape.
  Note the answer is **reversed**: the model emits the least significant digit first, so carries
  propagate in the same direction as generation. Writing it forwards asks the model to predict
  the leading digit before it has computed the carries that determine it.
- **Zero-padding** -- `05` not `5`, so digit positions mean the same thing in every example.
- **Number of digits** -- 2 is enough to have carries and small enough to enumerate exhaustively
  (100x100 = 10,000 pairs).

In [ ]:
VOCABULARY = {f"{i:d}": i for i in range(10)}
VOCABULARY["+"] = 10
VOCABULARY["="] = 11

# convenience aliases
TOKEN_TO_ID = VOCABULARY
ID_TO_TOKEN = {idx: token for token, idx in TOKEN_TO_ID.items()}
D_VOCAB = len(VOCABULARY)

# 1D (seq,) or 2D (batch, seq) ids, as a tensor or plain lists
TokenIds = Tensor | list[int] | list[list[int]]


def to_tokens(strings: str | list[str]) -> Int[Tensor, "batch seq"]:
    """'49+97=641' -> tensor([[4, 9, 10, 9, 7, 11, 6, 4, 1]]). Always returns (batch, seq)."""
    if isinstance(strings, str):
        strings = [strings]
    return t.tensor([[TOKEN_TO_ID[c] for c in s] for s in strings], device=device)


def to_str_tokens(tokens: TokenIds) -> list[str] | list[list[str]]:
    """Token ids -> the characters they stand for, one per token. Nested if the input is 2D."""
    if isinstance(tokens, Tensor):
        tokens = tokens.tolist()
    if tokens and isinstance(tokens[0], list):
        return [[ID_TO_TOKEN[i] for i in row] for row in tokens]
    return [ID_TO_TOKEN[i] for i in tokens]


def to_string(tokens: TokenIds) -> str | list[str]:
    """Token ids -> '49+97=641'. A list of strings if the input is 2D."""
    strs = to_str_tokens(tokens)
    if strs and isinstance(strs[0], list):
        return ["".join(row) for row in strs]
    return "".join(strs)


# sequences will have the result of the computation reversed


## 2. Dataset

Enumerate every `(a, b)` pair and split. No `datasets`, no Arrow, no DataLoader workers --
10,000 examples of 9 tokens fits in memory as a single tensor.

**The split fraction is the most consequential knob.** Hold out a large fraction and you may
reproduce **grokking**: the model memorises the training pairs, sits at chance on the held-out
ones, then abruptly generalises thousands of steps later. That is Nanda's phenomenon, and
producing it on this task is a result in itself. Worth running a sweep over the fraction.

Also decide: **compute loss only on the answer positions**. The question digits are given, so
predicting them is free and would dilute the metric.

In [ ]:
pairs = []

num_digits = 2
for i in range(10**num_digits):
    for j in range(10**num_digits):
        pairs.append(f"{i:0{num_digits}d}+{j:0{num_digits}d}={f'{(i+j):0{num_digits+1}d}'[::-1]}")

data = t.tensor([[TOKEN_TO_ID[token] for token in example] for example in pairs], device=device)
data

In [ ]:
seed = 42
frac_train = 0.3

g = t.Generator().manual_seed(seed)
# np.random.seed(seed)

_n = data.shape[0]
split = t.randperm(_n, generator=g)
train_idx = split[:int(frac_train * _n)].to(device=device)
test_idx = split[int(frac_train * _n):].to(device=device)


assert t.cat([train_idx, test_idx], dim=0).sort().values.equal(t.arange(len(data), device=data.device))

train = data[train_idx]
test = data[test_idx]
print(f"train: {train.shape} \ntest: {test.shape}")

## 3. Train

Use a `HookedTransformer` from TransformerLens rather than the `DemoTransformer` above.

You have already proven you can write the internals -- that was notebook 1. Building the
project model as a `HookedTransformer` gets you hook points for free and skips debugging a
`state_dict` mapping later. Keep `DemoTransformer` here as reference.

Sizing: this model should be *tiny*. 1-2 layers, small `d_model`, `d_vocab=12`. Training runs
in minutes, so iterate freely.

In [ ]:
for name, param in model.named_parameters():
    print(f"{name:<20} {str(param.shape):<20} {param.numel()}")
sum(p.numel() for p in model.parameters())      # 584,556

## 4. Interpretability

Roughly in order of cost:

- **Attention patterns** on carry vs non-carry examples (`circuitsvis`). Do specific heads look
  at the digit positions that generate a carry?
- **Per-head ablation** -- zero `hook_z` for each head in turn, measure the accuracy drop. This
  tells you *which* heads matter, not *how* they work.
- **Direct logit attribution** -- which component actually writes the answer digit.
- **The mechanism** -- how does carry information move between digit positions?

Reference: `chapter1_transformer_interp/exercises/part52_grokking_and_modular_arithmetic` is
ARENA's own exercise on the Nanda result, already downloaded. Read it before this phase.

Hook names and shapes: `notes/img/1_architecture/transformerlens-hook-names-annotated_cell059.png`.

**Watch out for attention sinks.** These sequences have no BOS token, so heads with nothing
useful to attend to will dump their attention on whatever is at position 0 or on the `=`.
A head that appears to "attend to the equals sign" for no clear reason is probably a sink,
not a circuit.

## 5. Write-up

`README.md` and `results/`, both already scaffolded in the repo. Motivation, method, findings,
plots. Compare the circuit you find against Nanda's modular-arithmetic one -- that contrast is
the point of the whole project.